# Step 1 — W8A16 Main Body Quantization

**Feature:** INT8 weights, FP16 activations (W8A16) on all main transformer Linear layers.

In [ ]:
SRC     = "/home/prashant.takale/icml/4_models/qwen-weights"             # FP16 base
OUT_DIR = "/home/prashant.takale/icml/4_models/qwen3.5-4b-w8a16-clean"   # output

NUM_CALIB   = 256
MAX_SEQ_LEN = 2048

import os
os.environ.setdefault("HF_HUB_OFFLINE", "0")
os.environ.setdefault("HF_DATASETS_OFFLINE", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("Paths set.", flush=True)

In [ ]:
# ── Load tokenizer + model 
from transformers import AutoModelForImageTextToText, AutoTokenizer

print("Loading tokenizer ...", flush=True)
tok = AutoTokenizer.from_pretrained(SRC, trust_remote_code=True)

print("Loading model on CPU (FP16 base) ...", flush=True)
model = AutoModelForImageTextToText.from_pretrained(
    SRC, torch_dtype="auto", trust_remote_code=True, device_map="cpu"
)
print("Model loaded.", flush=True)

In [ ]:
# ── Calibration data 
from datasets import load_dataset

print(f"Loading calibration data ({NUM_CALIB} ultrachat samples) ...", flush=True)
ds = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{NUM_CALIB}]")

def preprocess(ex):
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}
ds = ds.map(preprocess, remove_columns=ds.column_names)

def tokenize(ex):
    return tok(ex["text"], padding=False, truncation=True,
               max_length=MAX_SEQ_LEN, add_special_tokens=False)
ds = ds.map(tokenize, remove_columns=ds.column_names)
print(f"Calibration dataset ready: {len(ds)} samples", flush=True)

In [ ]:
# ── W8A16 GPTQ quantization 

# Targets all Linear layers EXCEPT:

#   lm_head               —  output projection (quality sensitive, kept FP16)
#   mtp.*                 —  MTP speculative head (quantized separately in step 3)
#   visual.* / vision.*   —  vision tower (removed in step 5)
#   linear_attn.*         —  GatedDeltaNet projections (quantized separately in step 4)


from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    targets="Linear",
    scheme="W8A16",
    ignore=[
        "lm_head",
        "re:.*mtp.*",
        "re:.*visual.*",
        "re:.*vision.*",
        "re:.*linear_attn.*",
    ],
)

print("Running GPTQ oneshot (sequential per-layer GPU onloading) ...", flush=True)
print("  This takes ~15-20 min on a single GPU.", flush=True)
oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_CALIB,
    output_dir=OUT_DIR,
)

In [ ]:
# ── Save tokenizer + processor 

tok.save_pretrained(OUT_DIR)
try:
    from transformers import AutoProcessor
    AutoProcessor.from_pretrained(SRC, trust_remote_code=True).save_pretrained(OUT_DIR)
    print("Processor saved.", flush=True)
except Exception as e:
    print(f"Processor save skipped: {e}", flush=True)

print(f"\n Step 1 DONE → {OUT_DIR}", flush=True)

In [ ]:
# ── Verify output 
    
import os, json
from safetensors import safe_open

sf = os.path.join(OUT_DIR, "model.safetensors")
idx = os.path.join(OUT_DIR, "model.safetensors.index.json")

files = []
if os.path.exists(idx):
    files = sorted(set(json.load(open(idx))["weight_map"].values()))
elif os.path.exists(sf):
    files = ["model.safetensors"]

packed, fp16 = 0, 0
for fn in files:
    with safe_open(os.path.join(OUT_DIR, fn), "pt") as f:
        for k in f.keys():
            if k.endswith(".weight_packed"): packed += 1
            elif k.endswith(".weight"): fp16 += 1

print(f"W8A16 packed layers : {packed}")
print(f"FP16 weight layers  : {fp16}")
print("(mtp.*, linear_attn.*, lm_head, visual.* should still be FP16 .weight)")